In [1]:
import pandas as pd

In [ ]:
import gradio as gr
import pandas as pd, numpy as np
import unicodedata
import os
import random
from gtts import gTTS
import asyncio
import edge_tts


VOICE = "en-GB-RyanNeural" # "en-US-GuyNeural"
VOICE = "en-US-GuyNeural"
RATE = "-45%"  #


# ---------------------------
# Cargar datos
# ---------------------------
df = pd.read_excel("traducciones.xlsx")

for col in ["correct_count", "wrong_count", "is_difficult", "needs_second_pass"]:
    if col not in df.columns:
        df[col] = 0

df = df.sort_values(by="freq", ascending=False).reset_index(drop=True)

# ---------------------------
# Preview DF
# ---------------------------
def get_df_preview(n=300):
    ax = df[df["is_difficult"] == 1]
    return ax.head(n)

# ---------------------------
# Constantes
# ---------------------------
COOLDOWN_CORRECT = 14
COOLDOWN_WRONG = 8
COOLDOWN_SECOND_PASS = 25

# ---------------------------
# Métricas
# ---------------------------
def get_metrics(df):
    mastered  = (df["correct_count"] >= 3).sum()
    learning  = (df["correct_count"] < 3).sum()
    difficult = (df["is_difficult"] == 1).sum()
    return mastered, learning, difficult

# ---------------------------
# Audio
# ---------------------------
# def generate_audio(word):
#     word = word.split(" | ")[1]
#     file = "audio.mp3"
#     gTTS(text=word, lang='en').save(file)
#     return file

def generate_audio(word):
    print("def generate_audio(word):", word)
    word = word.split(" | ")[1]

    if word in ["nan", np.nan]:
        word = "nan"

    file = "audio.mp3"
    asyncio.run(edge_tts.Communicate(word, VOICE, rate = RATE).save(file))
    return file

# ---------------------------
# Estado
# ---------------------------
def init_state():
    return {
        "cooldown": {},
        "word_counter": 0,
        "current": None,
    }

# ---------------------------
# Selección de palabra
# ---------------------------
def get_next_word(state, only_difficult):
    counter = state["word_counter"]
    candidates = []

    for _, row in df.iterrows():
        word = row["sentences2"] 

        if row["sentences2"] not in ["", "nan", np.nan]:

            if row["correct_count"] >= 3:
                continue

            if only_difficult and row["is_difficult"] != 1:
                continue

            if counter < state["cooldown"].get(word, 0):
                continue

            priority = row["freq"] + (row["wrong_count"] * 50)
            candidates.append((priority, row))

        else:
            continue

    if not candidates:
        state["current"] = None
        return state

    candidates.sort(key=lambda x: x[0], reverse=True)
    chosen = candidates[0][1]

    state["current"] = chosen.to_dict()
    state["word_counter"] += 1
    return state

# ---------------------------
# UI helpers
# ---------------------------
def format_word(word):
    return f"<div style='font-size:30px; text-align:center; font-weight:bold;'>{word}</div>"

def show_translation(state):
    if state["current"] is None:
        return ""
    return f"<div style='font-size:25px; text-align:center;'>📖 {state['current']['translation_sentences2']}</div>"

# ---------------------------
# RESPUESTAS
# ---------------------------
def answer_known(state, only_difficult):
    if state["current"] is None:
        return "⏳ Sin palabras", None, "", "", "", "", "", state, get_df_preview()

    word = state["current"]["sentences2"]
    print(word)
    idx = df.index[df["sentences2"] == word][0]
    counter = state["word_counter"]

    df.loc[idx, "correct_count"] += 1
    df.loc[idx, "wrong_count"] = 0

    if df.loc[idx, "needs_second_pass"] == 1:
        df.loc[idx, "correct_count"] = 3
    else:
        df.loc[idx, "needs_second_pass"] = 1
        state["cooldown"][word] = counter + COOLDOWN_SECOND_PASS

    state["cooldown"][word] = counter + COOLDOWN_CORRECT
    df.to_excel("traducciones.xlsx", index=False)

    state = get_next_word(state, only_difficult)

    if state["current"] is None:
        return "⏳ Fin", None, "", "", "", "", "", state, get_df_preview()

    next_word = state["current"]["sentences2"]
    audio = generate_audio(next_word)
    m, l, d = get_metrics(df)

    return format_word(next_word), audio, "", "", str(m), str(l), str(d), state, get_df_preview()

def answer_skip(state, only_difficult):
    if state["current"] is None:
        return "⏳ Sin palabras", None, "", "", "", "", "", state, get_df_preview()

    word = state["current"]["sentences2"]
    translation = state["current"]["translation_sentences2"]
    idx = df.index[df["sentences2"] == word][0]
    counter = state["word_counter"]

    df.loc[idx, "wrong_count"] += 1
    state["cooldown"][word] = counter + COOLDOWN_WRONG
    df.to_excel("traducciones.xlsx", index=False)

    translation_html = f"<div style='font-size:28px; text-align:center;'>❌ {translation}</div>"

    state = get_next_word(state, only_difficult)

    if state["current"] is None:
        return "⏳ Fin", None, "", translation_html, "", "", "", state, get_df_preview()

    next_word = state["current"]["sentences2"]
    audio = generate_audio(next_word)
    m, l, d = get_metrics(df)

    return format_word(next_word), audio, "", "", str(m), str(l), str(d), state, get_df_preview() # translation_html

def mark_difficult(state, only_difficult):
    if state["current"] is None:
        return "⏳ Not words", None, "", "", "", "", "", state, get_df_preview()

    word = state["current"]["sentences2"]
    idx = df.index[df["sentences2"] == word][0]

    df.loc[idx, "is_difficult"] = 1
    state["cooldown"][word] = 100
    df.loc[idx, "correct_count"] = 3
    df.to_excel("traducciones.xlsx", index=False)

    state = get_next_word(state, only_difficult)

    if state["current"] is None:
        return "⏳ End", None, "", "", "", "", "", state, get_df_preview()

    next_word = state["current"]["sentences2"]
    audio = generate_audio(next_word)
    m, l, d = get_metrics(df)

    return format_word(next_word), audio, "", "", str(m), str(l), str(d), state, get_df_preview()

def mark_mastered(state, only_difficult):
    if state["current"] is None:
        return "⏳ Not words", None, "", "", "", "", "", state, get_df_preview()

    word = state["current"]["sentences2"]
    idx = df.index[df["sentences2"] == word][0]

    df.loc[idx, "correct_count"] = 3
    df.to_excel("traducciones.xlsx", index=False)

    state = get_next_word(state, only_difficult)

    if state["current"] is None:
        return "⏳ End", None, "", "", "", "", "", state, get_df_preview()

    next_word = state["current"]["sentences2"]
    audio = generate_audio(next_word)
    m, l, d = get_metrics(df)

    return format_word(next_word), audio, "", "", str(m), str(l), str(d), state, get_df_preview()


def add_new_word(en_text):
    global df

    en_text = en_text.strip()

    if en_text == "":
        return " ", "⚠️ Required register word"

    if en_text in df["word"].values:
        return " ", "⚠️ Word exists in dataset."

    new_freq = 0.9999 #df["freq"].max() + 0.1 if len(df) > 0 else 1

    new_row = {
        "word": en_text,
        "freq": new_freq,
        "correct_count": 0,
        "wrong_count": 0,
        "is_difficult": 0,     
        "needs_second_pass": 0,
        "sentences":"nan",
        "translation_sentences":"nan",
        "sentences2":"nan | nan",
        "translation_sentences2":"nan | nan"
    }

    df.loc[len(df)] = new_row
    df.sort_values(by="freq", ascending=False, inplace=True)
    df.reset_index(drop=True, inplace=True)
    df.to_excel("traducciones.xlsx", index=False)

    return "", f"✅ Add done"

# ---------------------------
# START
# ---------------------------
def start_app(state, only_difficult):
    state = get_next_word(state, only_difficult)

    if state["current"] is None:
        return "⏳ Not words", None, "", "", "", "", "", state, get_df_preview()

    word = state["current"]["sentences2"]
    audio = generate_audio(word)
    m, l, d = get_metrics(df)

    return format_word(word), audio, "", "", str(m), str(l), str(d), state, get_df_preview()

# ---------------------------
# UI
# ---------------------------
with gr.Blocks(css="""
#btn_1 { background-color:#A2D668 !important; color:white; border-radius:10px; }
#btn_2 { background-color:#F77A1D !important; color:white; border-radius:10px; }
#btn_3 { background-color:#7030A0 !important; color:white; border-radius:10px; }
#btn_4 { background-color:#46B1E1 !important; color:white; border-radius:10px; }
#btn_5 {
    background-color: #FF6B6B !important;
    color: white !important;
    border-radius: 10px;
}
#btn_add { background-color:#2E7D32 !important; color:white; border-radius:10px; }
""") as app:

    state = gr.State(init_state())

    gr.Markdown("<h1 style='text-align:center;'>Train My English</h1>")

    only_difficult = gr.Checkbox(label="😈 Only difficult words")

    word = gr.Markdown()
    feedback = gr.Markdown()

    show_btn = gr.Button("Show translate", elem_id="btn_4")

    with gr.Row():
        known_btn = gr.Button("Known", elem_id="btn_1")
        skip_btn = gr.Button("Pass", elem_id="btn_2")
        master_btn = gr.Button("Dominated", elem_id="btn_3")
        difficult_btn = gr.Button("Difficult", elem_id="btn_5")

    audio = gr.Audio(autoplay=True)


    gr.Markdown("### ➕ Add new word")
    with gr.Row():
        add_word_en = gr.Textbox(label="Word", placeholder="")
    add_word_btn = gr.Button("➕ Add to dataset", elem_id="btn_add")
    add_word_feedback = gr.Markdown()


    mastered = gr.Textbox(label="🏆 Dominated")
    learning = gr.Textbox(label="📚 Learning")
    difficult = gr.Textbox(label="😈 Difficult")

    dummy1 = gr.Textbox(visible=False)


    df_view = gr.Dataframe(label="📊 Dataset", row_count=500, interactive=True)

    outputs = [word, audio, dummy1, feedback, mastered, learning, difficult, state, df_view]

    show_btn.click(show_translation, inputs=[state], outputs=feedback)
    known_btn.click(answer_known, inputs=[state, only_difficult], outputs=outputs)
    skip_btn.click(answer_skip, inputs=[state, only_difficult], outputs=outputs)
    master_btn.click(mark_mastered, inputs=[state, only_difficult], outputs=outputs)
    difficult_btn.click(mark_difficult, inputs=[state, only_difficult], outputs=outputs)

    add_word_btn.click(add_new_word, inputs=[add_word_en], outputs=[add_word_en, add_word_feedback])
    
    app.load(start_app, inputs=[state, only_difficult], outputs=outputs)

app.launch()

C:\Users\PC\AppData\Local\Temp\ipykernel_20240\2921671826.py:286: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css="""


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


def generate_audio(word): thus | Thus, the lesson was particularly enlightening today.
def generate_audio(word): his | His car is parked outside the building.
def generate_audio(word): her | She picked up the book from the table.
def generate_audio(word): them | They brought their lunch to school for lunch.
def generate_audio(word): its | The cat rubbed its nose against my hand.


Exception in callback _ProactorBasePipeTransport._call_connection_lost()
handle: <Handle _ProactorBasePipeTransport._call_connection_lost()>
Traceback (most recent call last):
  File "c:\Users\PC\miniconda3\Lib\asyncio\events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\PC\miniconda3\Lib\asyncio\proactor_events.py", line 165, in _call_connection_lost
    self._sock.shutdown(socket.SHUT_RDWR)
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^
ConnectionResetError: [WinError 10054] Se ha forzado la interrupción de una conexión existente por el host remoto


def generate_audio(word): than | I prefer coffee than tea.
def generate_audio(word): into | She carefully placed her book into her backpack.
def generate_audio(word): him | He gave me the book he found in the park.
def generate_audio(word): even | Even though it was raining heavily outside, she decided to go for a walk anyway.
